## Robustness of Persistent Recovery Patterns to Alternative Response Thresholds

### **Purpose**

#### This notebook evaluates whether the main conclusions from the exploratory recovery analysis depend strongly on the empirical response floor used to classify gene responses.

#### The primary analysis used the 95th percentile of pairwise control-replicate differences as a control-derived response floor. Here, the same recovery-classification procedure is repeated using more permissive and more stringent thresholds derived from the 90th, 95th, and 99th percentiles of control variation.

#### The analysis tests whether changing this threshold alters the main patterns identified in the project, including the proportion and ranking of persistent responses across stress conditions, the identity and breadth of persistent genes, the directional consistency of multi-stress persistence, and the GO-Slim functional interpretation of high-breadth persistent genes.

#### The aim is not to identify an optimal threshold, but to determine whether the main biological conclusions remain recognizable across reasonable alternative response-floor choices.

### Reconstructing and Testing Alternative Control-Derived Response Floors

#### The empirical response floor was recalculated from absolute expression differences between the three control replicates across the informative-gene set. In addition to the 95th-percentile threshold used in the primary exploratory analysis, the 90th and 99th percentiles were calculated as more permissive and more stringent alternatives.

#### These thresholds are used to test whether the main persistence results depend strongly on the exact percentile chosen for the control-derived response floor. The purpose is to evaluate sensitivity to a reasonable change in the exploratory threshold, not to select a new threshold based on which value produces the most favorable result.

In [1]:
from pathlib import Path
from itertools import combinations

import numpy as np
import pandas as pd

In [2]:
# Display settings
pd.set_option("display.max_columns", 20)
pd.set_option("display.max_rows", 30)
pd.set_option("display.precision", 3)

In [3]:
EXPRESSION_PATH = Path(
    "../data/raw/GSE226105_rLog-Normalized_counts.csv"
)

ROBUSTNESS_DIR = Path(
    "../data/interim/robustness_checks"
)

ROBUSTNESS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

In [4]:
expression_data = pd.read_csv(                                # loading and standardizing the expression matrix
    EXPRESSION_PATH
)

if "gene_id" not in expression_data.columns:
    expression_data = expression_data.rename(
        columns={
            "Unnamed: 0": "gene_id"
        }
    )

expression_data.shape

(28775, 58)

In [5]:
expression = (                                               # creating the informative-gene expression matrix
    expression_data
    .set_index("gene_id")
)

informative_expression = expression.loc[
    (expression != 0).any(axis=1)
]

informative_expression.shape

(26118, 57)

In [6]:
control_columns = [
    column
    for column in informative_expression.columns
    if column.startswith("CONT")
]

control_columns

['CONT 1', 'CONT 2', 'CONT 3']

In [7]:
control_pairwise_differences = []                             # calculating pairwise differences between control replicates

for sample_a, sample_b in combinations(
    control_columns,
    2
):
    differences = (                                           # rlog-expression difference for every informative gene
        informative_expression[sample_a]
        - informative_expression[sample_b]
    ).abs()

    control_pairwise_differences.append(
        differences
    )

control_pairwise_differences = pd.concat(
    control_pairwise_differences,
    ignore_index=True
)

len(control_pairwise_differences)

78354

In [8]:
response_floors = (                                            # calculating the alternative empirical response floors
    control_pairwise_differences
    .quantile(
        [
            0.90,
            0.95,
            0.99,
        ]
    )
)

response_floors

0.90    0.273
0.95    0.342
0.99    0.554
dtype: float64

### Reconstructing the Experimental Design and Calculating Condition-Level Expression Shifts

#### The experimental design was reconstructed directly from the expression-matrix column names, assigning each sample to its state, standardized stress condition, and biological replicate. The reconstructed manifest was then used to calculate gene-wise mean expression for the control, endpoint, and recovery groups.

#### Endpoint and recovery shifts were calculated relative to the mean control expression for each informative gene. These reconstructed shift matrices provide the common input for applying the alternative 90th-, 95th-, and 99th-percentile response floors during the robustness analysis.

In [9]:
import re                                                       # importing regex

In [10]:
condition_map = {                                               # standardizing the stress condition names
    "COLD HL": "cold_high_light",
    "COLD": "cold",
    "T": "heat_drought_high_light",
    "HEAT HL": "heat_high_light",
    "HEAT DROUGHT": "heat_drought",
    "HEAT": "heat",
    "DROUGHT": "drought",
    "Humidity": "high_humidity",
    "HL": "high_light",
}

In [11]:
manifest_rows = []                                               # reconstructing the sample manifest

for column in informative_expression.columns:

    if column.startswith("CONT"):

        replicate = int(
            re.search(r"(\d+)$", column).group(1)
        )

        manifest_rows.append(                                    # storing the control metadata
            {
                "matrix_column": column,
                "state": "control",
                "condition": "control",
                "replicate": replicate,
            }
        )

        continue

    if column.startswith("endpoint_"):                          # determining endpoint vs recovery
        state = "endpoint"

    elif column.startswith("recovery_"):
        state = "recovery"

    else:
        raise ValueError(
            f"Unrecognized sample column: {column}"
        )

    sample_body = column.split("_", 1)[1].strip()

    match = re.match(                                          # separating conditions from replicate numbers
        r"^(.*?)(?:\s+)?(\d+)$",
        sample_body
    )

    if match is None:
        raise ValueError(
            f"Could not parse sample: {column}"
        )

    raw_condition = match.group(1).strip()                     # standardizing the extracted condition
    replicate = int(match.group(2))

    manifest_rows.append(
        {
            "matrix_column": column,
            "state": state,
            "condition": condition_map[raw_condition],
            "replicate": replicate,
        }
    )

sample_manifest = pd.DataFrame(
    manifest_rows
)

sample_manifest.head()

,matrix_column,state,condition,replicate
0,endpoint_COLD HL 1,endpoint,cold_high_light,1
1,endpoint_COLD HL 2,endpoint,cold_high_light,2
2,endpoint_COLD HL 3,endpoint,cold_high_light,3
3,endpoint_COLD 1,endpoint,cold,1
4,endpoint_COLD 2,endpoint,cold,2


In [12]:
sample_manifest.groupby(                      # checking how many samples exist for every state-condition combination
    [
        "state",
        "condition",
    ]
).size()

state     condition              
control   control                    3
endpoint  cold                       3
          cold_high_light            3
          drought                    3
          heat                       3
          heat_drought               3
          heat_drought_high_light    3
          heat_high_light            3
          high_humidity              3
          high_light                 3
recovery  cold                       3
          cold_high_light            3
          drought                    3
          heat                       3
          heat_drought               3
          heat_drought_high_light    3
          heat_high_light            3
          high_humidity              3
          high_light                 3
dtype: int64

In [13]:
condition_order = [                                       # consistent stress-condition order
    "cold",
    "cold_high_light",
    "drought",
    "heat",
    "heat_drought",
    "heat_drought_high_light",
    "heat_high_light",
    "high_humidity",
    "high_light",
]

In [14]:
control_columns = sample_manifest.loc[                    # calculating mean control expression       
    sample_manifest["state"] == "control",
    "matrix_column"
].tolist()

control_mean = (
    informative_expression[
        control_columns
    ]
    .mean(axis=1)
)

In [15]:
endpoint_means = {}                                       # calculating mean endpoint expression by stress

for condition in condition_order:

    columns = sample_manifest.loc[
        (sample_manifest["state"] == "endpoint")
        & (sample_manifest["condition"] == condition),
        "matrix_column"
    ].tolist()

    endpoint_means[condition] = (
        informative_expression[
            columns
        ]
        .mean(axis=1)
    )

endpoint_means = pd.DataFrame(
    endpoint_means
)

In [16]:
recovery_means = {}                                          # calculating mean recovery expression by stress

for condition in condition_order:

    columns = sample_manifest.loc[
        (sample_manifest["state"] == "recovery")
        & (sample_manifest["condition"] == condition),
        "matrix_column"
    ].tolist()

    recovery_means[condition] = (
        informative_expression[
            columns
        ]
        .mean(axis=1)
    )

recovery_means = pd.DataFrame(
    recovery_means
)

### Calculating Endpoint and Recovery Shifts Relative to Control

In [17]:
endpoint_shift = (                                     # endpoint shift = endpoint mean − control mean
    endpoint_means
    .sub(
        control_mean,
        axis=0
    )
)

recovery_shift = (                                    # recovery shift = recovery mean − control mean
    recovery_means
    .sub(
        control_mean,
        axis=0
    )
)

In [18]:
endpoint_shift.shape

(26118, 9)

In [19]:
recovery_shift.shape

(26118, 9)

In [20]:
sample_manifest.groupby("state").size()

state
control      3
endpoint    27
recovery    27
dtype: int64

### **Observation —** Reconstruction of the Experimental Design

#### The reconstructed sample manifest contained the expected 57 samples: 3 controls, 27 endpoint samples, and 27 recovery samples, with three biological replicates for each stress-state combination.

#### The resulting endpoint and recovery shift matrices each contained 26,118 informative genes across nine stress conditions, confirming that the experimental design and condition-level expression summaries were reconstructed correctly for the robustness analysis.

### Reconstructing and Validating the Primary Recovery Classification

#### The recovery-classification logic was implemented as reusable functions so that the same rules can be applied unchanged at each response-floor threshold. Gene-condition combinations with a qualifying endpoint response are classified as recovered, persistent in the same direction, or substantially reversed according to the magnitude and direction of the recovery shift.

#### The original 95th-percentile response floor was applied first to verify that the reconstructed procedure reproduces the primary recovery classification before alternative thresholds are tested. This establishes the 95th-percentile result as the baseline against which the 90th- and 99th-percentile sensitivity results can be compared.

In [21]:
def classify_recovery_profiles(                              # defining the recovery profile classification function
    endpoint_shift,                                          # how far each gene is from control at endpoint
    recovery_shift,                                          # how far each gene is from control after recovery
    response_floor,                                          # the minimum absolute difference required to count as a qualifying response
):
    categories = pd.DataFrame(
        pd.NA,
        index=endpoint_shift.index,
        columns=endpoint_shift.columns,
        dtype="object",
    )

    for condition in endpoint_shift.columns:                # processing one stress condition at a time

        endpoint = endpoint_shift[condition]
        recovery = recovery_shift[condition]

        endpoint_eligible = (                               # whether the endpoint response qualifies
            endpoint.abs() >= response_floor
        )

        recovery_residual = (                               # whether a substantial residual response remains at recovery
            recovery.abs() >= response_floor
        )

        same_direction = (                                  # whether endpoint and recovery are on the same side of control
            endpoint * recovery > 0
        )

        opposite_direction = (                              # reversal across control
            endpoint * recovery < 0
        )

# Assigning the three recovery categories

        categories.loc[                                     # recovered
            endpoint_eligible
            & ~recovery_residual,
            condition,
        ] = "recovered"

        categories.loc[                                     # persistent same direction
            endpoint_eligible
            & recovery_residual
            & same_direction,
            condition,
        ] = "persistent_same_direction"

        categories.loc[                                    # substantial reversal
            endpoint_eligible
            & recovery_residual
            & opposite_direction,
            condition,
        ] = "substantial_reversal"

    return categories

In [22]:
def summarize_recovery_categories(                          # defining the recovery category summary function
    recovery_categories,
):
    rows = []                                               # one summary row per condition

    for condition in recovery_categories.columns:

        values = recovery_categories[condition]

        rows.append(
            {
                "condition": condition,
                "endpoint_eligible": (
                    values.notna().sum()
                ),
                "recovered": (
                    values.eq("recovered").sum()
                ),
                "persistent_same_direction": (
                    values.eq(
                        "persistent_same_direction"
                    ).sum()
                ),
                "substantial_reversal": (
                    values.eq(
                        "substantial_reversal"
                    ).sum()
                ),
            }
        )

    return pd.DataFrame(rows)

In [ ]:
# Reproducing the primary 95th percentile classification

original_floor = response_floors.loc[0.95]                    # the floor

recovery_categories_95 = (                                    # reclassifying every gene-condition combination using the floor
    classify_recovery_profiles(
        endpoint_shift=endpoint_shift,
        recovery_shift=recovery_shift,
        response_floor=original_floor,
    )
)

recovery_summary_95 = (
    summarize_recovery_categories(
        recovery_categories_95
    )
)

recovery_summary_95

,condition,endpoint_eligible,recovered,persistent_same_direction,substantial_reversal
0,cold,8915,8838,71,6
1,cold_high_light,12210,11677,187,346
2,drought,11942,10613,1240,89
3,heat,1310,1226,70,14
4,heat_drought,14088,13690,344,54
5,heat_drought_high_light,8609,8323,255,31
6,heat_high_light,814,729,72,13
7,high_humidity,1712,1661,28,23
8,high_light,1950,1154,779,17


In [24]:
recovery_summary_95["category_total"] = (             # validating the recovery-category
    recovery_summary_95["recovered"]
    + recovery_summary_95[
        "persistent_same_direction"
    ]
    + recovery_summary_95[
        "substantial_reversal"
    ]
)

recovery_summary_95[                                 # whether the three recovery categories account for every classified endpoint-eligible gene
    "categories_sum_to_endpoint_eligible"
] = (
    recovery_summary_95["category_total"]
    == recovery_summary_95["endpoint_eligible"]
)

recovery_summary_95

,condition,endpoint_eligible,recovered,persistent_same_direction,substantial_reversal,category_total,categories_sum_to_endpoint_eligible
0,cold,8915,8838,71,6,8915,True
1,cold_high_light,12210,11677,187,346,12210,True
2,drought,11942,10613,1240,89,11942,True
3,heat,1310,1226,70,14,1310,True
4,heat_drought,14088,13690,344,54,14088,True
5,heat_drought_high_light,8609,8323,255,31,8609,True
6,heat_high_light,814,729,72,13,814,True
7,high_humidity,1712,1661,28,23,1712,True
8,high_light,1950,1154,779,17,1950,True


### **Insight —** Reproduction of the Primary Recovery Classification

#### Before testing alternative response thresholds, the recovery-classification procedure was reconstructed directly from the original rlog expression matrix using the original 95th-percentile control-derived response floor of 0.342263 rlog units.

#### The resulting counts of qualifying endpoint responses, recovered genes, persistent same-direction genes, and substantial reversals matched the original `exploratory_analysis` notebook results exactly across all nine stress conditions. For every condition, the three recovery categories also summed exactly to the number of qualifying endpoint responses.

#### This shows that the robustness notebook reproduces the original classification procedure correctly. Any changes observed when using the 90th- or 99th-percentile response floors can therefore be attributed to the change in threshold rather than to differences in sample grouping, expression processing, or recovery-classification logic.

### Testing the Sensitivity of Recovery Classification to Alternative Response Floors

#### The recovery classification was repeated using control-derived response floors at the 90th, 95th, and 99th percentiles while keeping the expression data, sample grouping, and classification rules unchanged. For each threshold, the numbers of endpoint-eligible, recovered, persistent same-direction, and substantially reversed gene-condition responses were summarized.

#### Because changing the response floor also changes the number of genes that qualify at endpoint, both absolute persistent-gene counts and category fractions among qualifying endpoint responders were compared. This allows the robustness analysis to distinguish changes caused simply by having more or fewer eligible genes from changes in the overall recovery pattern.

In [25]:
floor_specs = {                                                # defining the three response floors
    "p90": response_floors.loc[0.90],                          # p90 -> 90th-percentile control-derived floor
    "p95": response_floors.loc[0.95],                          # p95 -> original 95th-percentile floor
    "p99": response_floors.loc[0.99],                          # p99 -> 99th-percentile control-derived floor
}

# Applying the recovery classification at each threshold

recovery_categories_by_floor = {}                              # store the complete gene-by-condition classification table for each response floor
robustness_summary_tables = []                                 # store the summarized counts and proportions for each floor

for floor_label, response_floor in floor_specs.items():

    categories = classify_recovery_profiles(
        endpoint_shift=endpoint_shift,
        recovery_shift=recovery_shift,
        response_floor=response_floor,
    )

    summary = summarize_recovery_categories(
        categories
    )

    summary["floor_label"] = floor_label
    summary["response_floor"] = response_floor

    summary["persistent_fraction_of_responders"] = (           # calculating recovery-category fraction
        summary["persistent_same_direction"]
        / summary["endpoint_eligible"]
    )

    summary["recovered_fraction_of_responders"] = (
        summary["recovered"]
        / summary["endpoint_eligible"]
    )

    summary["reversal_fraction_of_responders"] = (
        summary["substantial_reversal"]
        / summary["endpoint_eligible"]
    )

    recovery_categories_by_floor[
        floor_label
    ] = categories

    robustness_summary_tables.append(
        summary
    )

response_floor_sensitivity = pd.concat(
    robustness_summary_tables,
    ignore_index=True,
)

response_floor_sensitivity

,condition,endpoint_eligible,recovered,persistent_same_direction,substantial_reversal,floor_label,response_floor,persistent_fraction_of_responders,recovered_fraction_of_responders,reversal_fraction_of_responders
0,cold,10802,10638,148,16,p90,0.273,0.014,0.985,1.481e-03
1,cold_high_light,13578,12544,405,629,p90,0.273,0.030,0.924,4.632e-02
2,drought,13347,11270,1934,143,p90,0.273,0.145,0.844,1.071e-02
3,heat,2268,2124,118,26,p90,0.273,0.052,0.937,1.146e-02
4,heat_drought,15092,14528,480,84,p90,0.273,0.032,0.963,5.566e-03
5,heat_drought_high_light,10670,10194,418,58,p90,0.273,0.039,0.955,5.436e-03
6,heat_high_light,1663,1509,130,24,p90,0.273,0.078,0.907,1.443e-02
7,high_humidity,2721,2603,61,57,p90,0.273,0.022,0.957,2.095e-02
8,high_light,3405,2233,1134,38,p90,0.273,0.333,0.656,1.116e-02
9,cold,8915,8838,71,6,p95,0.342,0.008,0.991,6.730e-04


### Comparing Persistent Gene Counts Across Thresholds

In [26]:
persistent_count_sensitivity = (
    response_floor_sensitivity
    .pivot(
        index="condition",
        columns="floor_label",
        values="persistent_same_direction",
    )
    .reindex(condition_order)
)

persistent_count_sensitivity

floor_label,p90,p95,p99
condition,,,
cold,148,71,25
cold_high_light,405,187,44
drought,1934,1240,497
heat,118,70,25
heat_drought,480,344,129
heat_drought_high_light,418,255,78
heat_high_light,130,72,21
high_humidity,61,28,9
high_light,1134,779,361


### Comparing The Fraction of Responders That Remained Persistent

In [27]:
persistent_fraction_sensitivity = (
    response_floor_sensitivity
    .pivot(
        index="condition",
        columns="floor_label",
        values="persistent_fraction_of_responders",
    )
    .reindex(condition_order)
)

persistent_fraction_sensitivity = (
    persistent_fraction_sensitivity * 100
)

persistent_fraction_sensitivity.round(2)

floor_label,p90,p95,p99
condition,,,
cold,1.37,0.80,0.60
cold_high_light,2.98,1.53,0.56
drought,14.49,10.38,5.96
heat,5.20,5.34,5.91
heat_drought,3.18,2.44,1.14
heat_drought_high_light,3.92,2.96,1.93
heat_high_light,7.82,8.85,11.48
high_humidity,2.24,1.64,1.49
high_light,33.30,39.95,62.24


### Comparing the Fraction of Responders That Recover

In [28]:
recovered_fraction_sensitivity = (
    response_floor_sensitivity
    .pivot(
        index="condition",
        columns="floor_label",
        values="recovered_fraction_of_responders",
    )
    .reindex(condition_order)
)

recovered_fraction_sensitivity = (
    recovered_fraction_sensitivity * 100
)

recovered_fraction_sensitivity.round(2)

floor_label,p90,p95,p99
condition,,,
cold,98.48,99.14,99.40
cold_high_light,92.38,95.63,98.87
drought,84.44,88.87,93.78
heat,93.65,93.59,93.14
heat_drought,96.26,97.17,98.72
heat_drought_high_light,95.54,96.68,98.02
heat_high_light,90.74,89.56,88.52
high_humidity,95.66,97.02,97.52
high_light,65.58,59.18,37.24


### **Observation —** Response-Floor Sensitivity

#### Changing the response floor produced substantial changes in the absolute numbers of qualifying endpoint responses and persistent same-direction genes.

#### For high light, the persistent fraction increased from 33.3% at p90 to 40.0% at p95 and 62.2% at p99. Drought showed the opposite trend, decreasing from 14.5% to 10.4% to 6.0%, while heat+high light increased from 7.8% to 8.9% to 11.5%. Heat remained comparatively stable at approximately 5–6% across all three thresholds.

#### Cold, cold+high light, heat+drought, heat+drought+high light, and high humidity remained dominated by recovered responses across the tested thresholds.

#### Absolute persistent-gene counts decreased as the response floor became more stringent. For example, drought decreased from 1,934 to 1,240 to 497 persistent genes, high light from 1,134 to 779 to 361, and heat+drought from 480 to 344 to 129 across p90, p95, and p99, respectively.

### **Insight —** Persistence Patterns Remain Recognizable Across Response-Floor Choices

#### Changing the empirical response floor substantially altered the absolute numbers of genes classified as endpoint-responsive and persistent, as expected. The 90th-percentile threshold was more permissive (0.272532 rlog units), the original 95th-percentile threshold was intermediate (0.342263), and the 99th-percentile threshold was more stringent (0.554173).

#### Despite these changes in gene counts, the main differences in recovery behavior among stress conditions remained recognizable.

#### High light showed the highest proportional persistence at every threshold. Persistent same-direction genes represented 33.3% of qualifying endpoint responses at p90, 40.0% at p95, and 62.2% at p99. This shows that the strong persistence associated with high light was not created by the original p95 cutoff and became proportionally stronger when the analysis was restricted to larger expression displacements.

#### Drought also remained among the more persistent conditions, although its persistent fraction decreased from 14.5% at p90 to 10.4% at p95 and 6.0% at p99. In contrast, heat+high light increased from 7.8% to 8.9% to 11.5%, while heat remained relatively stable at approximately 5–6% across all three thresholds.

#### Cold, cold+high light, heat+drought, heat+drought+high light, and high humidity remained predominantly recovery-oriented under all three response-floor choices.

#### The central pattern from the primary analysis therefore remained intact: a condition with many qualifying endpoint responses did not necessarily retain a large persistent fraction after recovery. Heat+drought retained only 3.2%, 2.4%, and 1.1% of qualifying endpoint responses as persistent across p90, p95, and p99, whereas high light retained 33.3%, 40.0%, and 62.2%, respectively.

#### Overall, the main stress-specific recovery pattern was robust to substantial changes in the empirical response floor, supporting the conclusion that the observed differences in persistence are not simply a consequence of choosing the 95th-percentile threshold.

### Testing the Stability of Stress-Condition Persistence Rankings Across Response Floors

#### Spearman rank correlations were used to evaluate whether the relative ordering of stress conditions by persistent fraction remained stable across the 90th-, 95th-, and 99th-percentile response floors. This complements the direct comparison of persistence percentages by testing whether stresses that were relatively more persistent under one threshold also tended to remain relatively more persistent under another.

#### The actual condition ranks were also calculated for each threshold so that changes in ordering could be inspected directly. Because only nine stress conditions are available, the rank correlations and observed ranking patterns are interpreted primarily as robustness measures rather than as standalone inferential tests.

In [29]:
from scipy.stats import spearmanr                          # Spearman rank-correlation test

In [30]:
rank_stability_rows = []

floor_pairs = [
    ("p90", "p95"),
    ("p95", "p99"),
    ("p90", "p99"),
]

for floor_a, floor_b in floor_pairs:                        # calculating the Spearman rank correlations

    rho, p_value = spearmanr(
        persistent_fraction_sensitivity[floor_a],
        persistent_fraction_sensitivity[floor_b],
    )

    rank_stability_rows.append(
        {
            "comparison": f"{floor_a}_vs_{floor_b}",
            "spearman_rho": rho,
            "p_value": p_value,
        }
    )

rank_stability = pd.DataFrame(
    rank_stability_rows
)

rank_stability

,comparison,spearman_rho,p_value
0,p90_vs_p95,0.983,1.936e-06
1,p95_vs_p99,0.950,8.763e-05
2,p90_vs_p99,0.900,9.431e-04


In [31]:
persistent_fraction_ranks = (                            # calculating the actual persistence ranks
    persistent_fraction_sensitivity
    .rank(
        ascending=False,
        method="min",
    )
    .astype(int)
)

persistent_fraction_ranks

floor_label,p90,p95,p99
condition,,,
cold,9,9,8
cold_high_light,7,8,9
drought,2,2,3
heat,4,4,4
heat_drought,6,6,7
heat_drought_high_light,5,5,5
heat_high_light,3,3,2
high_humidity,8,7,6
high_light,1,1,1


### **Observation —** Stability of Persistence Rankings Across Response Floors

#### Spearman rank correlations between the persistent-fraction rankings were high across all three response-floor comparisons:

#### - p90 vs p95: ρ = 0.983

#### - p95 vs p99: ρ = 0.950

#### - p90 vs p99: ρ = 0.900

#### The highest-persistence conditions also showed very stable rankings across thresholds. High light remained rank 1 at p90, p95, and p99. Drought ranked 2, 2, and 3, while heat+high light ranked 3, 3, and 2. Heat remained rank 4 at all three thresholds, and heat+drought+high light remained rank 5.

#### Larger rank changes were mainly confined to the lower-persistence conditions, where the persistent fractions were already relatively small.

### **Insight —** Stress-Condition Ranking Is Highly Robust to the Response Floor

#### The relative ranking of stress conditions by persistent fraction was strongly preserved when the response floor was changed.

#### Spearman rank correlations were 0.983 between the p90 and p95 analyses, 0.950 between p95 and p99, and 0.900 even between the most permissive and most stringent thresholds. These high correlations show that the relative ordering of the stress conditions changed very little as the response floor became more restrictive.

#### The top of the ranking was particularly stable. High light ranked first at all three thresholds. Drought ranked second at p90 and p95 and third at p99, while heat+high light ranked third, third, and second. Heat remained fourth and heat+drought+high light remained fifth at every threshold.

#### The lower-persistence conditions changed position somewhat, but their persistent fractions remained small throughout. Overall, the stress-specific persistence ranking was highly stable across the tested response floors, showing that the main ordering observed in the primary analysis does not depend on the precise choice of the original 95th-percentile threshold.

### Testing the Stability of Persistent Gene Identity Across Response Floors

#### Persistent same-direction genes were collected separately for each stress condition at the p90, p95, and p99 response floors. The resulting gene sets were compared to determine whether stricter thresholds retained subsets of the genes identified under more permissive thresholds.

#### Gene-set overlap was quantified using the Jaccard index, and retention percentages were calculated to show how many persistent genes remained classified as persistent as the response floor became more stringent. This analysis tests the stability of the actual persistent-gene identities, complementing the earlier comparisons of counts, proportions, and stress-condition rankings.

In [32]:
persistent_sets_by_floor = {}                         # creating persistent gene sets for each response floor

for floor_label, categories in (
    recovery_categories_by_floor.items()
):

    persistent_sets_by_floor[floor_label] = {}

    for condition in condition_order:

        persistent_sets_by_floor[
            floor_label
        ][condition] = set(
            categories.index[
                categories[condition]
                == "persistent_same_direction"
            ]
        )

### Checking Whether Stricter Persistent-Gene Sets Are Nested Within More Permissive Sets

In [33]:
nesting_rows = []

for condition in condition_order:

    p90_genes = persistent_sets_by_floor[
        "p90"
    ][condition]

    p95_genes = persistent_sets_by_floor[
        "p95"
    ][condition]

    p99_genes = persistent_sets_by_floor[
        "p99"
    ][condition]

    nesting_rows.append(
        {
            "condition": condition,
            "p95_subset_of_p90": (
                p95_genes.issubset(p90_genes)
            ),
            "p99_subset_of_p95": (
                p99_genes.issubset(p95_genes)
            ),
            "p99_subset_of_p90": (
                p99_genes.issubset(p90_genes)
            ),
        }
    )

persistent_set_nesting = pd.DataFrame(
    nesting_rows
)

persistent_set_nesting

,condition,p95_subset_of_p90,p99_subset_of_p95,p99_subset_of_p90
0,cold,True,True,True
1,cold_high_light,True,True,True
2,drought,True,True,True
3,heat,True,True,True
4,heat_drought,True,True,True
5,heat_drought_high_light,True,True,True
6,heat_high_light,True,True,True
7,high_humidity,True,True,True
8,high_light,True,True,True


### Measuring Persistent-Gene Set Similarity with the Jaccard Index

In [34]:
gene_set_stability_rows = []

floor_pairs = [
    ("p90", "p95"),
    ("p95", "p99"),
    ("p90", "p99"),
]

for condition in condition_order:

    for floor_a, floor_b in floor_pairs:

        genes_a = persistent_sets_by_floor[
            floor_a
        ][condition]

        genes_b = persistent_sets_by_floor[
            floor_b
        ][condition]

        intersection = genes_a & genes_b                               # intersection of the two sets
        union = genes_a | genes_b                                      # union of the two sets

        gene_set_stability_rows.append(
            {
                "condition": condition,
                "comparison": (
                    f"{floor_a}_vs_{floor_b}"
                ),
                "n_a": len(genes_a),
                "n_b": len(genes_b),
                "shared_genes": len(intersection),
                "jaccard": (                                             # jaccard similarity
                    len(intersection) / len(union)
                    if len(union) > 0
                    else np.nan
                ),
            }
        )

gene_set_stability = pd.DataFrame(
    gene_set_stability_rows
)

gene_set_stability

,condition,comparison,n_a,n_b,shared_genes,jaccard
0,cold,p90_vs_p95,148,71,71,0.480
1,cold,p95_vs_p99,71,25,25,0.352
2,cold,p90_vs_p99,148,25,25,0.169
3,cold_high_light,p90_vs_p95,405,187,187,0.462
4,cold_high_light,p95_vs_p99,187,44,44,0.235
5,cold_high_light,p90_vs_p99,405,44,44,0.109
6,drought,p90_vs_p95,1934,1240,1240,0.641
7,drought,p95_vs_p99,1240,497,497,0.401
8,drought,p90_vs_p99,1934,497,497,0.257
9,heat,p90_vs_p95,118,70,70,0.593


In [35]:
gene_set_jaccard = (                              # comparing the gene-set stability among stress conditions
    gene_set_stability
    .pivot(
        index="condition",
        columns="comparison",
        values="jaccard",
    )
)

gene_set_jaccard.round(3)

comparison,p90_vs_p95,p90_vs_p99,p95_vs_p99
condition,,,
cold,0.480,0.169,0.352
cold_high_light,0.462,0.109,0.235
drought,0.641,0.257,0.401
heat,0.593,0.212,0.357
heat_drought,0.717,0.269,0.375
heat_drought_high_light,0.610,0.187,0.306
heat_high_light,0.554,0.162,0.292
high_humidity,0.459,0.148,0.321
high_light,0.687,0.318,0.463


### Calculating How Many Persistent Genes Survive Stricter Thresholds

In [36]:
retention_rows = []

for condition in condition_order:

    p90 = persistent_sets_by_floor[
        "p90"
    ][condition]

    p95 = persistent_sets_by_floor[
        "p95"
    ][condition]

    p99 = persistent_sets_by_floor[
        "p99"
    ][condition]

    retention_rows.append(
        {
            "condition": condition,
            "p95_retained_from_p90": (
                len(p95) / len(p90)
                if len(p90) > 0
                else np.nan
            ),
            "p99_retained_from_p95": (
                len(p99) / len(p95)
                if len(p95) > 0
                else np.nan
            ),
            "p99_retained_from_p90": (
                len(p99) / len(p90)
                if len(p90) > 0
                else np.nan
            ),
        }
    )

persistent_gene_retention = pd.DataFrame(
    retention_rows
)

for column in [
    "p95_retained_from_p90",
    "p99_retained_from_p95",
    "p99_retained_from_p90",
]:
    persistent_gene_retention[column] *= 100

persistent_gene_retention.round(1)

,condition,p95_retained_from_p90,p99_retained_from_p95,p99_retained_from_p90
0,cold,48.0,35.2,16.9
1,cold_high_light,46.2,23.5,10.9
2,drought,64.1,40.1,25.7
3,heat,59.3,35.7,21.2
4,heat_drought,71.7,37.5,26.9
5,heat_drought_high_light,61.0,30.6,18.7
6,heat_high_light,55.4,29.2,16.2
7,high_humidity,45.9,32.1,14.8
8,high_light,68.7,46.3,31.8


### **Insight —** Stricter Thresholds Progressively Trim the Persistent Sets Rather Than Replacing Them

#### For every stress condition, the persistent gene sets were perfectly nested across response-floor choices:

#### p99 persistent genes ⊆ p95 persistent genes ⊆ p90 persistent genes.

#### This nesting is expected from the classification rule because a gene that exceeds a stricter response floor for both endpoint and recovery displacement will also exceed a lower floor. It should therefore be treated primarily as a consistency check rather than as independent evidence of biological robustness.

#### The retention analysis nevertheless shows how strongly the persistent sets are reduced as the criterion becomes more stringent. Between p90 and p95, retention ranged from approximately 46% to 72% across conditions. Drought retained 64.1% of its p90 persistent genes, high light retained 68.7%, and heat+drought retained 71.7%. Moving from p95 to the much stricter p99 floor retained 40.1% of drought genes and 46.3% of high-light genes, while several smaller persistent sets were reduced more strongly.

#### Because the sets are nested, the Jaccard similarities largely reflect the fraction of the broader set retained at the stricter threshold. The main interpretation is therefore that increasing the response-floor stringency progressively narrows each persistent set to genes that continue to satisfy the stronger response criterion, rather than replacing them with an unrelated collection of genes.

### Testing the Robustness of Persistence Breadth and Direction Across Response Floors

#### Persistence breadth was recalculated separately at the p90, p95, and p99 response floors by counting the number of stress conditions in which each gene was classified as persistent in the same direction. The numbers of genes reaching increasing breadth levels were then compared to determine whether multi-stress persistence remained evident under alternative response criteria.

#### For genes persistent under at least two stress conditions, the direction of the recovery displacement relative to control was also compared across stresses. Genes were classified as directionally consistent when all of their persistent responses remained on the same side of control. The analysis additionally distinguished genes that were consistently above control from those consistently below control.

#### This tests whether two features identified in the primary analysis — multi-stress persistence and the strong directional bias among broadly persistent genes — remain recognizable when the empirical response floor is changed.

In [37]:
persistence_breadth_by_floor = {}                       # calculating persistence breadth at each response floor

for floor_label, categories in (
    recovery_categories_by_floor.items()
):

    persistent_matrix = (
        categories
        .eq("persistent_same_direction")
    )

    breadth = persistent_matrix.sum(axis=1)             # calculating the breadth

    persistence_breadth_by_floor[
        floor_label
    ] = breadth

In [38]:
breadth_distribution_95 = (                            # inspecting original p95 breadth distribution
    persistence_breadth_by_floor["p95"]
    .loc[
        persistence_breadth_by_floor["p95"] > 0
    ]
    .value_counts()
    .sort_index()
)

breadth_distribution_95

1    883
2    350
3    170
4    126
5     48
6     17
7     13
8      2
Name: count, dtype: int64

In [39]:
breadth_summary_rows = []                            # comparing the persistence breadth across thresholds

for floor_label, breadth in (
    persistence_breadth_by_floor.items()
):

    breadth_summary_rows.append(
        {
            "floor_label": floor_label,              # genes persistent under different stress/stresses

            "persistent_at_least_1": (               # atleast one stress
                breadth >= 1
            ).sum(),

            "persistent_at_least_2": (               # atleast two stresses
                breadth >= 2
            ).sum(),

            "persistent_at_least_3": (               # atleast three stresses
                breadth >= 3
            ).sum(),

            "persistent_at_least_4": (               # atleast four stresses
                breadth >= 4
            ).sum(),

            "persistent_at_least_5": (               # atleast five stresses
                breadth >= 5
            ).sum(),

            "persistent_at_least_6": (               # atleast six stresses
                breadth >= 6
            ).sum(),

            "maximum_breadth": breadth.max(),        # largest number of stress conditions under which any gene remain persistent
        }
    )

breadth_sensitivity_summary = pd.DataFrame(
    breadth_summary_rows
)

breadth_sensitivity_summary

,floor_label,persistent_at_least_1,persistent_at_least_2,persistent_at_least_3,persistent_at_least_4,persistent_at_least_5,persistent_at_least_6,maximum_breadth
0,p90,2534,1124,576,344,149,63,9
1,p95,1609,726,376,206,80,32,8
2,p99,647,305,143,62,21,9,7


### Testing Directional Consistency Among Multi-Stress Persistent Genes

In [40]:
direction_robustness_rows = []

for floor_label, categories in (
    recovery_categories_by_floor.items()
):

    persistent_matrix = (                                   # persistence gene matrix
        categories
        .eq("persistent_same_direction")
    )

    breadth = persistent_matrix.sum(axis=1)                 # counting the persistence breadth

    persistent_direction = (                                # direction of recovery shift
        np.sign(recovery_shift)
        .where(persistent_matrix)                           # this geneshave already been classified persistent_same_direction
    )


    direction_min = (                                       # minimun direction
        persistent_direction.min(axis=1)
    )

    direction_max = (                                       # maximum direction
        persistent_direction.max(axis=1)
    )

    multi_stress = breadth >= 2                             # genes persistent under atleast two stress conditions

    direction_consistent = (                                # identifying genes that are directionally consistent
        multi_stress
        & (direction_min == direction_max)
    )

    consistently_up = (                                     # consistently above control
        direction_consistent
        & (direction_max == 1)
    )

    consistently_down = (                                   # consistently below control
        direction_consistent
        & (direction_max == -1)
    )

    inconsistent = (                                        # directionally inconsistent
        multi_stress
        & ~direction_consistent
    )

    n_multi = int(multi_stress.sum())
    n_consistent = int(
        direction_consistent.sum()
    )
    n_up = int(
        consistently_up.sum()
    )
    n_down = int(
        consistently_down.sum()
    )
    n_inconsistent = int(
        inconsistent.sum()
    )

    direction_robustness_rows.append(
        {
            "floor_label": floor_label,
            "multi_stress_persistent_genes": n_multi,
            "directionally_consistent": n_consistent,
            "directionally_inconsistent": n_inconsistent,
            "consistent_up": n_up,
            "consistent_down": n_down,
            "percent_directionally_consistent": (
                100 * n_consistent / n_multi                    # percentage of persistent genes (>= 2 stresses) that shows consistent direction
                if n_multi > 0
                else np.nan
            ),
            "percent_consistent_genes_up": (
                100 * n_up / n_consistent                       # percentage of consistent genes that are above control
                if n_consistent > 0
                else np.nan
            ),
        }
    )

direction_robustness = pd.DataFrame(
    direction_robustness_rows
)

direction_robustness.round(2)

,floor_label,multi_stress_persistent_genes,directionally_consistent,directionally_inconsistent,consistent_up,consistent_down,percent_directionally_consistent,percent_consistent_genes_up
0,p90,1124,1102,22,1082,20,98.04,98.19
1,p95,726,720,6,711,9,99.17,98.75
2,p99,305,304,1,303,1,99.67,99.67


### **Insight —** A Multi-Stress Persistent Component Remains Across Permissive and Conservative Thresholds

#### The size of the persistent gene population decreased substantially as the response floor became more stringent, but a multi-stress persistent component remained present at every threshold.

#### At the p90 floor, 2,534 genes were persistent in at least one stress condition and 344 were persistent in at least four. Under the primary p95 definition, these numbers fell to 1,609 and 206, respectively. Even at the conservative p99 floor, 647 genes remained persistent in at least one stress and 62 remained persistent in at least four.

#### The same pattern was evident across alternative persistence-breadth definitions. At p95, 376 genes were persistent in at least three conditions, 206 in at least four, and 80 in at least five. The original definition of persistence in four or more stresses therefore represents an intermediate operational choice: it is more selective than three-condition recurrence while retaining enough genes for downstream functional analysis.

#### The maximum observed persistence breadth was threshold-dependent. At p90, at least one gene satisfied the persistent criterion across all nine stress conditions, whereas the maximum breadth was eight at p95 and seven at p99. Statements about the exact maximum number of stresses in which an individual gene is persistent therefore depend on the chosen response floor, whereas the broader finding that a recurrent multi-stress persistent component exists is much more stable.

#### The strong directional pattern observed in the primary analysis remained almost unchanged across alternative response floors. Between 98.0% and 99.7% of genes persistent under multiple stress conditions showed the same expression direction across those stresses, and almost all of these directionally consistent genes remained above control.

#### This shows that the predominance of persistent elevation among multi-stress persistent genes is not dependent on the original p95 response floor. The pattern becomes, if anything, more concentrated as the response criterion becomes more stringent

### **Interim Robustustness Conclusion**

#### The sensitivity analysis shows that the exact number of persistent genes depends, as expected, on the empirical response floor. However, the main patterns identified in the primary p95 analysis are considerably more stable than the raw gene counts alone would suggest.

#### Across response floors spanning the 90th to 99th percentiles of control variation, the ranking of stress conditions remained highly correlated, high light consistently showed the greatest proportional persistence, and conditions with many qualifying endpoint responses did not necessarily retain a large persistent fraction after recovery. A recurrent multi-stress persistent component remained detectable, and the strong directional consistency of multi-stress persistence—predominantly above control—was preserved.

#### The primary p95 analysis can therefore be interpreted as one operational definition of a broader pattern that remains recognizable across substantially more permissive and more stringent response floors, rather than as a result that depends uniquely on a single threshold choice.

### Preparing GO-Slim Annotations and High-Breadth Persistent Gene Sets for Functional Robustness Testing

#### Arabidopsis GO-Slim annotations were loaded and reduced to unique gene–GO-aspect–GO-Slim associations. High-breadth persistent gene sets were then defined separately at the p90, p95, and p99 response floors using the same criterion as the primary analysis: persistence in at least four stress conditions.

#### These gene sets will be used to test whether the broad functional patterns associated with high-breadth persistence remain similar when the empirical response floor is changed.

In [41]:
GO_SLIM_PATH = Path(
    "../data/external/arabidopsis_annotation/ATH_GO_GOSLIM.txt"
)

In [42]:
go_columns = [                                # annotation-file column names
    "gene_id",
    "tair_object_id",
    "object_name",
    "go_relationship",
    "go_term",
    "go_id",
    "go_numeric_id",
    "go_aspect",
    "go_slim_term",
    "evidence_code",
    "evidence_detail",
    "evidence_with",
    "reference",
    "annotator",
    "annotation_date",
]

In [43]:
go_annotations = pd.read_csv(
    GO_SLIM_PATH,
    sep="\t",
    header=None,
    names=go_columns,
    dtype=str,
    comment="!",
)

go_annotations.shape

(337091, 15)

In [44]:
go_slim_pairs = (                           # gene-to-GO slim mapping
    go_annotations[
        [
            "gene_id",
            "go_aspect",
            "go_slim_term",
        ]
    ]
    .dropna(
        subset=[
            "gene_id",
            "go_aspect",
            "go_slim_term",
        ]
    )
    .drop_duplicates()
)

go_slim_pairs.shape

(188875, 3)

In [45]:
go_slim_pairs[
    [
        "go_aspect",
        "go_slim_term",
    ]
].drop_duplicates().shape

(98, 2)

### Creating High-Breadth Persistent Gene Sets at Each Response Floor

In [46]:
high_breadth_sets = {}

for floor_label, breadth in (
    persistence_breadth_by_floor.items()
):

    high_breadth_sets[floor_label] = set(
        breadth.index[
            breadth >= 4
        ]
    )

In [47]:
{
    floor_label: len(genes)                    # checking the size of the three sets
    for floor_label, genes
    in high_breadth_sets.items()
}

{'p90': 344, 'p95': 206, 'p99': 62}

### Reconstructing GO-Slim Enrichment for the Primary High-Breadth Persistent Gene Set

#### GO-Slim enrichment was reconstructed for the high-breadth persistent gene set defined at the primary p95 response floor. Genes persistent in at least four stress conditions were used as the foreground, while informative genes with available annotations for each GO aspect provided the corresponding background.

#### For each GO-Slim term, overrepresentation in the high-breadth set was tested using a one-sided Fisher exact test. Enrichment ratios were calculated to describe the relative representation of each term, and Benjamini-Hochberg correction was applied across the tested GO-Slim terms to control the false discovery rate.

#### This p95 analysis provides the primary functional reference against which GO-Slim enrichment at the alternative p90 and p99 response floors can subsequently be compared.

In [48]:
from scipy.stats import fisher_exact                               # Fisher's exact test
from statsmodels.stats.multitest import multipletests              # multiple testing correction

In [49]:
def run_go_slim_enrichment(
    foreground_ids,                                                  # genes persistent in >= 4 stresses (biological group interest)
    background_ids,                                                  # informative genes from the expression dataset
    go_slim_pairs,                                                   # connecting the genes associated with their GO-Slim terms
):
    rows = []

    for aspect in ["P", "F", "C"]:                                   # the three gene ontology aspects

        aspect_pairs = (
            go_slim_pairs.loc[
                go_slim_pairs["go_aspect"] == aspect
            ]
        )

        annotatable_background = (                                   # defining the annotable background
            set(aspect_pairs["gene_id"])
            & set(background_ids)
        )

        annotatable_foreground = (                                   # defining the annotable foreground
            set(foreground_ids)
            & annotatable_background
        )

        background_total = len(                                      # counting the effective background
            annotatable_background
        )

        foreground_total = len(                                      # counting the effective foreground
            annotatable_foreground
        )

        terms = (                                                    # examining every GO-Slim term
            aspect_pairs["go_slim_term"]
            .drop_duplicates()
        )

        for term in terms:                                           # testing each term separately

            term_genes = set(
                aspect_pairs.loc[
                    aspect_pairs["go_slim_term"]
                    == term,
                    "gene_id",
                ]
            )

            background_with_term = len(
                annotatable_background
                & term_genes
            )

            foreground_with_term = len(
                annotatable_foreground
                & term_genes
            )
        
            a = foreground_with_term                                 # foreground genes that have the GO-Slim term

            b = (                                                    # foreground genes that do not have the GO-Slim term
                foreground_total
                - foreground_with_term
            )

            c = (                                                    # non-foreground background genes that have the GO-Slim term
                background_with_term
                - foreground_with_term
            )

            d = (                                                    # non-foreground background genes that do not have the GO-Slim term
                background_total
                - foreground_total
                - c
            )

            odds_ratio, p_value = (                                  # one-sided fisher's exact test
                fisher_exact(
                    [
                        [a, b],
                        [c, d],
                    ],
                    alternative="greater",
                )
            )

            foreground_fraction = (                                  # foreground frequency
                a / foreground_total
                if foreground_total > 0
                else np.nan
            )

            background_fraction = (                                  # background frequency
                background_with_term
                / background_total
                if background_total > 0
                else np.nan
            )

            enrichment_ratio = (                                     # enrichment ratio
                foreground_fraction
                / background_fraction
                if background_fraction > 0
                else np.nan
            )

            rows.append(
                {
                    "go_aspect": aspect,
                    "go_slim_term": term,
                    "foreground_with_term": a,
                    "foreground_total": foreground_total,
                    "background_with_term": (
                        background_with_term
                    ),
                    "background_total": (
                        background_total
                    ),
                    "enrichment_ratio": (
                        enrichment_ratio
                    ),
                    "odds_ratio": odds_ratio,
                    "p_value": p_value,
                }
            )

    enrichment = pd.DataFrame(rows)

    enrichment["fdr_bh"] = multipletests(                            # Benjamini-Hochberg multiple-testing FDR correction
        enrichment["p_value"],
        method="fdr_bh",
    )[1]

    return enrichment

In [50]:
background_ids = set(                                                # defining the expression analysis background
    informative_expression.index
)

### GO-Slim Enrichment for the Primary p95 Gene Set

In [51]:
go_slim_p95 = run_go_slim_enrichment(
    foreground_ids=high_breadth_sets["p95"],
    background_ids=background_ids,
    go_slim_pairs=go_slim_pairs,
)

In [52]:
go_slim_p95.shape

(98, 10)

### Selecting Statistically Significant GO-Slim Enrichments

In [53]:
go_slim_p95_significant = (
    go_slim_p95.loc[
        go_slim_p95["fdr_bh"] < 0.05
    ]
    .sort_values("fdr_bh")
)

len(go_slim_p95_significant)

24

In [54]:
go_slim_p95_significant[
    [
        "go_aspect",
        "go_slim_term",
        "foreground_with_term",
        "foreground_total",
        "background_with_term",
        "background_total",
        "enrichment_ratio",
        "fdr_bh",
    ]
].head(15)

,go_aspect,go_slim_term,foreground_with_term,foreground_total,background_with_term,background_total,enrichment_ratio,fdr_bh
16,P,response to biotic stimulus,74,168,1316,20524,6.870,1.503e-41
13,P,response to external stimulus,74,168,1410,20524,6.412,9.063e-40
15,P,response to stress,108,168,3678,20524,3.587,5.093e-39
25,P,response to chemical,91,168,2878,20524,3.863,2.954e-33
20,P,response to abiotic stimulus,59,168,1836,20524,3.926,8.627e-20
30,P,response to endogenous stimulus,41,168,1530,20524,3.274,1.274e-10
82,C,plasma membrane,52,206,3054,25866,2.138,9.863e-07
26,P,signal transduction,37,168,1822,20524,2.481,2.151e-06
39,P,cell death,10,168,220,20524,5.553,1.530e-04
86,C,extracellular region,42,206,2758,25866,1.912,2.724e-04


### **Observation -** Checking the newly constructed primary high-breadth gene set

#### All critical checkpoints match:

#### - GO annotation table: 337,091 × 15

#### - Unique GO-slim gene-term pairs: 188,875 × 3

#### - Unique aspect-term combinations: 98

#### - ≥4-stress foreground sizes: 344 / 206 / 62

#### - p95 enrichment tests: 98

#### - Significant p95 GO-slim terms at BH FDR < 0.05: 24

#### And the actual top terms are the same ones interpreted originally: response to biotic stimulus, external stimulus, stress, chemical and abiotic stimuli, signal transduction, plasma membrane, cell wall, kinase activity, secondary metabolism, light response etc.

### **Insight —** Reproduction of the Primary GO-Slim Enrichment

#### The p95 GO-Slim enrichment analysis reproduced the results from the primary biological-interpretation notebook. This confirms that the robustness analysis is using the same annotation source, background definition, enrichment test, and multiple-testing correction as the original analysis.

#### Any differences observed when the enrichment analysis is repeated at the p90 or p99 response floors can therefore be attributed to changes in the high-breadth persistent-gene set rather than to changes in the GO annotations, background gene universe, or statistical procedure.

### Testing the Robustness of GO-Slim Functional Enrichment Across Response Floors

#### GO-Slim enrichment was repeated for the high-breadth persistent gene sets defined at the p90, p95, and p99 response floors using the same annotation source, background universe, Fisher exact test, and Benjamini-Hochberg correction.

#### The overlap among FDR-significant GO-Slim terms was compared across response floors. In addition, each term significant in the primary p95 analysis was tracked across all three thresholds using both its enrichment ratio and FDR-adjusted significance.

#### This distinction is important because the high-breadth foreground decreases substantially as the response floor becomes more stringent. Functional robustness is therefore evaluated not only by whether a term remains formally significant, but also by whether its enrichment remains in the same direction and of similar magnitude across thresholds.

In [55]:
go_slim_enrichment_by_floor = {}                            # GO Slim enrichment at p90, p95 and p99
go_slim_significant_by_floor = {}

for floor_label in [
    "p90",
    "p95",
    "p99",
]:

    enrichment = run_go_slim_enrichment(
        foreground_ids=high_breadth_sets[
            floor_label
        ],
        background_ids=background_ids,
        go_slim_pairs=go_slim_pairs,
    )

    significant = (
        enrichment.loc[
            enrichment["fdr_bh"] < 0.05
        ]
        .sort_values("fdr_bh")
        .reset_index(drop=True)
    )

    go_slim_enrichment_by_floor[
        floor_label
    ] = enrichment

    go_slim_significant_by_floor[
        floor_label
    ] = significant

In [56]:
go_slim_significant_counts = pd.DataFrame(                   # comparing the number of significant GO-Slim terms
    {
        "floor_label": [
            "p90",
            "p95",
            "p99",
        ],
        "high_breadth_genes": [
            len(
                high_breadth_sets[
                    "p90"
                ]
            ),
            len(
                high_breadth_sets[
                    "p95"
                ]
            ),
            len(
                high_breadth_sets[
                    "p99"
                ]
            ),
        ],
        "significant_go_slim_terms": [
            len(
                go_slim_significant_by_floor[
                    "p90"
                ]
            ),
            len(
                go_slim_significant_by_floor[
                    "p95"
                ]
            ),
            len(
                go_slim_significant_by_floor[
                    "p99"
                ]
            ),
        ],
    }
)

go_slim_significant_counts

,floor_label,high_breadth_genes,significant_go_slim_terms
0,p90,344,27
1,p95,206,24
2,p99,62,14


In [57]:
significant_term_sets = {}                                   # sets of sognificant functional terms

for floor_label, df in (
    go_slim_significant_by_floor.items()
):

    significant_term_sets[
        floor_label
    ] = set(
        zip(
            df["go_aspect"],
            df["go_slim_term"],
        )
    )

In [58]:
{
    floor_label: len(terms)
    for floor_label, terms
    in significant_term_sets.items()
}

{'p90': 27, 'p95': 24, 'p99': 14}

### Measuring Overlap Among Significant Functional Terms

In [59]:
functional_overlap_rows = []

floor_pairs = [
    ("p90", "p95"),
    ("p95", "p99"),
    ("p90", "p99"),
]

for floor_a, floor_b in floor_pairs:

    terms_a = significant_term_sets[
        floor_a
    ]

    terms_b = significant_term_sets[
        floor_b
    ]

    shared = terms_a & terms_b                              # the number of significant terms shared by both analyses
    union = terms_a | terms_b                               # the total number of unique significant terms across them

    functional_overlap_rows.append(
        {
            "comparison": (
                f"{floor_a}_vs_{floor_b}"
            ),
            "n_terms_a": len(terms_a),
            "n_terms_b": len(terms_b),
            "shared_significant_terms": (
                len(shared)
            ),
            "jaccard": (                                    # jaccard similarity
                len(shared) / len(union)
                if len(union) > 0
                else np.nan
            ),
        }
    )

go_slim_functional_overlap = (
    pd.DataFrame(
        functional_overlap_rows
    )
)

go_slim_functional_overlap.round(3)

,comparison,n_terms_a,n_terms_b,shared_significant_terms,jaccard
0,p90_vs_p95,27,24,22,0.759
1,p95_vs_p99,24,14,13,0.520
2,p90_vs_p99,27,14,13,0.464


### Tracking the Primary p95 GO-Slim Terms Across All Three Floors

In [60]:
primary_terms = (                                         # GO-Slim terms that were significant in the primary p95 analysis
    go_slim_significant_by_floor[
        "p95"
    ][
        [
            "go_aspect",
            "go_slim_term",
        ]
    ]
    .copy()
)

term_sensitivity_tables = []

for floor_label in [
    "p90",
    "p95",
    "p99",
]:

    df = (
        go_slim_enrichment_by_floor[
            floor_label
        ][
            [
                "go_aspect",
                "go_slim_term",
                "foreground_with_term",
                "foreground_total",
                "enrichment_ratio",
                "fdr_bh",
            ]
        ]
        .copy()
    )

    df = df.rename(
        columns={
            "foreground_with_term":
                f"{floor_label}_foreground_with_term",
            "foreground_total":
                f"{floor_label}_foreground_total",
            "enrichment_ratio":
                f"{floor_label}_enrichment_ratio",
            "fdr_bh":
                f"{floor_label}_fdr_bh",
        }
    )

    term_sensitivity_tables.append(
        df
    )

In [61]:
primary_term_sensitivity = (                         
    primary_terms
)

for df in term_sensitivity_tables:

    primary_term_sensitivity = (                           # merging the results onto the primary p95 terms
        primary_term_sensitivity.merge(
            df,
            on=[
                "go_aspect",
                "go_slim_term",
            ],
            how="left",
            validate="one_to_one",
        )
    )

In [62]:
for floor_label in [
    "p90",
    "p95",
    "p99",
]:

    primary_term_sensitivity[                              # whether each primary term remains significant
        f"{floor_label}_significant"
    ] = (
        primary_term_sensitivity[
            f"{floor_label}_fdr_bh"
        ] < 0.05
    )

In [63]:
primary_term_sensitivity[
    [
        "go_aspect",
        "go_slim_term",
        "p90_enrichment_ratio",
        "p90_fdr_bh",
        "p90_significant",
        "p95_enrichment_ratio",
        "p95_fdr_bh",
        "p95_significant",
        "p99_enrichment_ratio",
        "p99_fdr_bh",
        "p99_significant",
    ]
].round(4)

,go_aspect,go_slim_term,p90_enrichment_ratio,p90_fdr_bh,p90_significant,p95_enrichment_ratio,p95_fdr_bh,p95_significant,p99_enrichment_ratio,p99_fdr_bh,p99_significant
0,P,response to biotic stimulus,6.408,0.000e+00,True,6.870,0.000e+00,True,9.357,0.000e+00,True
1,P,response to external stimulus,5.981,0.000e+00,True,6.412,0.000e+00,True,8.734,0.000e+00,True
2,P,response to stress,3.328,0.000e+00,True,3.587,0.000e+00,True,3.906,0.000e+00,True
3,P,response to chemical,3.371,0.000e+00,True,3.863,0.000e+00,True,4.279,0.000e+00,True
4,P,response to abiotic stimulus,3.293,0.000e+00,True,3.926,0.000e+00,True,3.354,4.000e-04,True
5,P,response to endogenous stimulus,2.780,0.000e+00,True,3.274,0.000e+00,True,2.683,2.550e-02,True
6,C,plasma membrane,1.994,0.000e+00,True,2.138,0.000e+00,True,2.322,6.200e-03,True
7,P,signal transduction,2.499,0.000e+00,True,2.481,0.000e+00,True,2.704,9.400e-03,True
8,P,cell death,5.767,0.000e+00,True,5.553,2.000e-04,True,5.598,9.480e-02,False
9,C,extracellular region,1.908,0.000e+00,True,1.912,3.000e-04,True,2.420,6.200e-03,True


In [64]:
primary_term_robustness_summary = {                           # significant GO-Slim terms that existed in the primary p95 analysis
    "primary_p95_terms": len(
        primary_term_sensitivity
    ),

    "enriched_above_1_at_all_floors": (                       # primary p95 significant terms that remained enriched at every response floor
        (
            primary_term_sensitivity[
                [
                    "p90_enrichment_ratio",
                    "p95_enrichment_ratio",
                    "p99_enrichment_ratio",
                ]
            ] > 1
        )
        .all(axis=1)
        .sum()
    ),

    "enriched_at_least_2x_at_all_floors": (                   # primary terms that remain at least twofold enriched at p90, p95, and p99
        (
            primary_term_sensitivity[
                [
                    "p90_enrichment_ratio",
                    "p95_enrichment_ratio",
                    "p99_enrichment_ratio",
                ]
            ] >= 2
        )
        .all(axis=1)
        .sum()
    ),

    "significant_at_all_floors": (                            # p95 significant terms that remain FDR-significant at all three thresholds
        primary_term_sensitivity[
            [
                "p90_significant",
                "p95_significant",
                "p99_significant",
            ]
        ]
        .all(axis=1)
        .sum()
    ),
}

primary_term_robustness_summary

{'primary_p95_terms': 24,
 'enriched_above_1_at_all_floors': np.int64(23),
 'enriched_at_least_2x_at_all_floors': np.int64(12),
 'significant_at_all_floors': np.int64(12)}

### **Insight —** Broad Functional Interpretation Remains Stable Across Response-Floor Choices

#### The broad functional interpretation of the high-breadth persistent gene set remained substantially stable when the empirical response floor was varied.

#### Using the same definition of persistence in at least four stress conditions produced 344 high-breadth genes at the p90 response floor, 206 at p95, and 62 at p99. Despite this large reduction in foreground size, the GO-Slim analysis identified 27, 24, and 14 significant terms, respectively.

#### The significant-term sets showed substantial overlap. Twenty-two of the 24 terms significant in the primary p95 analysis were also significant at p90, while 13 remained significant at the more stringent p99 threshold. Jaccard similarity of the significant-term sets was 0.759 between p90 and p95 and 0.520 between p95 and p99.

#### More importantly, the strongest biological themes remained enriched even when some individual terms lost formal significance with the smaller p99 foreground. Of the 24 primary p95 significant terms, 23 remained enriched above background at all three response floors, 12 remained at least two-fold enriched at every floor, and 12 remained statistically significant at every floor.

#### The most stable functions included responses to biotic, external, abiotic, chemical, endogenous, and general stress stimuli, together with signal transduction, plasma membrane, extracellular region, cell wall, secondary metabolism, and cytosolic localization. Several of these terms showed stronger enrichment ratios within the stricter p99 persistent set, indicating that the smaller foreground retained many of the same broad environmental-response functions.

#### Not every GO-Slim term was equally robust. Transporter activity, cell communication, response to light stimulus, kinase activity, and cell death lost formal significance at the strictest response floor, while signaling receptor activity was absent from the p99 foreground. These terms should therefore be interpreted as features of the primary p95 result rather than as properties that remain unchanged under every plausible response-floor definition.

#### Overall, the sensitivity analysis supports the central functional conclusion of `biological_interpretation` notebook: broadly persistent genes are consistently enriched for environmental-response, signaling, cellular-interface, and stress-associated functions. The exact set of significant GO-Slim terms changes with the response floor, but the main functional interpretation does not depend uniquely on the original p95 cutoff.

### Export Robustness-Analysis Results

#### The main results from the response-floor sensitivity, persistence-ranking, persistent-gene stability, persistence-breadth, directional-consistency, and GO-Slim functional-robustness analyses were exported to the dedicated robustness-check directory. Complete GO-Slim enrichment tables and FDR-significant subsets were also saved separately for each response floor.

In [65]:
# Condition-level response-floor sensitivity
response_floor_sensitivity.to_csv(
    ROBUSTNESS_DIR / "response_floor_sensitivity.csv",
    index=False,
)

persistent_count_sensitivity.reset_index().to_csv(
    ROBUSTNESS_DIR / "persistent_count_sensitivity.csv",
    index=False,
)

persistent_fraction_sensitivity.reset_index().to_csv(
    ROBUSTNESS_DIR / "persistent_fraction_sensitivity.csv",
    index=False,
)

recovered_fraction_sensitivity.reset_index().to_csv(
    ROBUSTNESS_DIR / "recovered_fraction_sensitivity.csv",
    index=False,
)

# Ranking and gene-set stability
rank_stability.to_csv(
    ROBUSTNESS_DIR / "persistent_fraction_rank_stability.csv",
    index=False,
)

persistent_fraction_ranks.reset_index().to_csv(
    ROBUSTNESS_DIR / "persistent_fraction_ranks.csv",
    index=False,
)

persistent_set_nesting.to_csv(
    ROBUSTNESS_DIR / "persistent_set_nesting.csv",
    index=False,
)

persistent_gene_retention.to_csv(
    ROBUSTNESS_DIR / "persistent_gene_retention.csv",
    index=False,
)

# Persistence breadth and direction
breadth_sensitivity_summary.to_csv(
    ROBUSTNESS_DIR / "persistence_breadth_sensitivity.csv",
    index=False,
)

direction_robustness.to_csv(
    ROBUSTNESS_DIR / "directional_consistency_sensitivity.csv",
    index=False,
)

# Functional robustness
go_slim_significant_counts.to_csv(
    ROBUSTNESS_DIR / "go_slim_significant_counts_by_floor.csv",
    index=False,
)

go_slim_functional_overlap.to_csv(
    ROBUSTNESS_DIR / "go_slim_significant_term_overlap.csv",
    index=False,
)

primary_term_sensitivity.to_csv(
    ROBUSTNESS_DIR / "primary_go_slim_term_sensitivity.csv",
    index=False,
)

In [66]:
for floor_label, enrichment in (
    go_slim_enrichment_by_floor.items()
):

    enrichment.to_csv(
        ROBUSTNESS_DIR
        / f"go_slim_enrichment_{floor_label}_all.csv",
        index=False,
    )

In [67]:
for floor_label, significant in (
    go_slim_significant_by_floor.items()
):

    significant.to_csv(
        ROBUSTNESS_DIR
        / f"go_slim_enrichment_{floor_label}_significant.csv",
        index=False,
    )

In [68]:
sorted(
    path.name
    for path in ROBUSTNESS_DIR.iterdir()
    if path.is_file()
)

['directional_consistency_sensitivity.csv',
 'go_slim_enrichment_p90_all.csv',
 'go_slim_enrichment_p90_significant.csv',
 'go_slim_enrichment_p95_all.csv',
 'go_slim_enrichment_p95_significant.csv',
 'go_slim_enrichment_p99_all.csv',
 'go_slim_enrichment_p99_significant.csv',
 'go_slim_significant_counts_by_floor.csv',
 'go_slim_significant_term_overlap.csv',
 'persistence_breadth_sensitivity.csv',
 'persistent_count_sensitivity.csv',
 'persistent_fraction_rank_stability.csv',
 'persistent_fraction_ranks.csv',
 'persistent_fraction_sensitivity.csv',
 'persistent_gene_retention.csv',
 'persistent_set_nesting.csv',
 'primary_go_slim_term_sensitivity.csv',
 'recovered_fraction_sensitivity.csv',
 'response_floor_sensitivity.csv']